# 滑动窗口专题笔记

**主题**：固定窗口 & 可变窗口  
**日期**：2026-06-08  
**题目**：LC 643 - LC 1004

---

## 滑动窗口通用框架

### 触发条件
- 题目涉及 **连续子数组 / 子串**
- 要求最大 / 最小 / 恰好满足某条件的子区间
- 暴力解是 O(n^2)，可以用窗口状态维护优化到 O(n)

### 两种模式

| 模式 | 窗口大小 | 典型题 |
|------|----------|--------|
| 固定窗口 | 固定为 k | LC643, LC239 |
| 可变窗口 | 动态收缩 | LC3, LC76, LC1004 |

### 通用模板（可变窗口）

```python
left = 0
state = ...   # 窗口内的状态
ans = ...

for right in range(len(nums)):
    state = update(state, nums[right])   # 1. 扩展窗口
    while not valid(state):              # 2. 收缩窗口
        state = remove(state, nums[left])
        left += 1
    ans = max(ans, right - left + 1)     # 3. 更新答案

return ans
```

---

## LC 643 - 子数组最大平均数 I

### 基本信息

| 项目 | 内容 |
|------|------|
| **题型** | 固定大小滑动窗口 |
| **难度** | 简单 |
| **链接** | https://leetcode.com/problems/maximum-average-subarray-i/ |

### 数据范围
- `n == nums.length`
- `1 <= k <= n <= 10^5`
- `-10^4 <= nums[i] <= 10^4`
- 答案误差小于 `1e-5` 均视为正确

### 算法思路
1. 对前 k 个元素求和，作为初始窗口
2. 滑动窗口向右：加入右侧新元素，减去左侧移出元素
3. 维护滑动过程中的最大和
4. 最终返回 `max_sum / k`

**关键点**：只需维护「和」而不是「平均数」，避免每步都做除法

### 复杂度
- **时间**：O(n)
- **空间**：O(1)

In [ ]:
# LC 643 -- 子数组最大平均数 I
from typing import List

class Solution:
    def findMaxAverage(self, nums: List[int], k: int) -> float:
        n_len = len(nums)
        # Step 1: 初始化第一个窗口
        ans = sum(nums[:k])
        max_sum = ans
        # Step 2: 右进左出
        for i in range(k, n_len):
            ans += nums[i] - nums[i - k]
            if ans > max_sum:
                max_sum = ans
        return max_sum / k

sol = Solution()
print(sol.findMaxAverage([1, 12, -5, -6, 50, 3], 4))  # 12.75
print(sol.findMaxAverage([5], 1))                      # 5.0

### 固定窗口模板

```python
window_val = sum(nums[:k])
ans = window_val
for i in range(k, len(nums)):
    window_val += nums[i] - nums[i - k]   # 右进左出
    ans = max(ans, window_val)
return ans
```

### 易错点
- 循环从 `k` 开始而非 `0`
- 移除的是 `nums[i - k]`，不是 `nums[i - k - 1]`

---

## LC 1004 - 最大连续 1 的个数 III

### 基本信息

| 项目 | 内容 |
|------|------|
| **题型** | 可变大小滑动窗口（条件约束收缩） |
| **难度** | 中等 |
| **链接** | https://leetcode.com/problems/max-consecutive-ones-iii/ |

### 数据范围
- `1 <= nums.length <= 10^5`
- `nums[i]` 只为 `0` 或 `1`
- `0 <= k <= nums.length`

### 算法思路

**关键转化**：「最多翻转 k 个 0」等价于「窗口内 0 的个数 <= k」

不需要真的翻转，只需维护窗口内 0 的计数：
1. `right` 右移扩展窗口，遇到 0 则 `zero_count += 1`
2. 当 `zero_count > k` 时，`left` 右移收缩直到合法
3. 每步更新最大窗口长度

**这个转化思路是可变窗口的核心**：把操作次数约束 → 窗口内某类元素的计数约束

### 复杂度
- **时间**：O(n) -- left、right 各最多移动 n 步
- **空间**：O(1)

In [ ]:
# LC 1004 -- 最大连续 1 的个数 III
from typing import List

class Solution:
    def longestOnes(self, nums: List[int], k: int) -> int:
        left = 0
        max_len = 0
        zero_count = 0

        for right in range(len(nums)):
            # Step 1: 扩展窗口
            if nums[right] == 0:
                zero_count += 1
            # Step 2: 条件不满足则收缩
            while zero_count > k:
                if nums[left] == 0:
                    zero_count -= 1
                left += 1
            # Step 3: 更新最大长度
            max_len = max(max_len, right - left + 1)

        return max_len

sol = Solution()
print(sol.longestOnes([1,1,1,0,0,0,1,1,1,1,0], k=2))                    # 6
print(sol.longestOnes([0,0,1,1,0,0,1,1,1,0,1,1,0,0,0,1,1,1,1], k=3))   # 10

### 可变窗口收缩模板（条件约束型）

```python
left = 0
counter = 0
ans = 0

for right in range(len(nums)):
    if condition(nums[right]):
        counter += 1
    while counter > limit:       # 超出限制则收缩
        if condition(nums[left]):
            counter -= 1
        left += 1
    ans = max(ans, right - left + 1)

return ans
```

### 同类题目

| 题号 | 题目 | 约束的「计数」 |
|------|------|----------------|
| LC 1004 | 最大连续1的个数 III | 窗口内 0 的个数 <= k |
| LC 424 | 替换后的最长重复字符 | 窗口内非最高频字符数 <= k |
| LC 1208 | 尽可能使字符串相等 | 窗口内转换总成本 <= maxCost |

### 易错点
- `while` 而非 `if`：收缩条件可能需要移动多步才满足
- 收缩时先判断 `nums[left]` 再 `left += 1`，顺序不能反
- `k=0` 边界情况：窗口内一遇到 0 就立即收缩，逻辑同样成立无需特判

---

## 两题对比总结

| | LC 643 | LC 1004 |
|---|---|---|
| **窗口类型** | 固定大小（k） | 可变大小 |
| **收缩触发** | 不需要收缩 | `zero_count > k` |
| **维护状态** | 窗口和 | 窗口内 0 的个数 |
| **答案更新** | 每步 | 收缩后每步 |
| **核心思路** | 右进左出维护固定和 | 操作次数 -> 计数约束的转化 |

## 记忆口诀

> **固定窗口**：初始化 -> 右进左出 -> 更新答案  
> **可变窗口**：右扩状态 -> 违规就收缩 -> 更新答案